# پروژه پایانی ۱: دیده‌بان نماد (سطح پایه)
**هدف پروژه:** 
 بررسی وضعیت چند سهم در بازار و تشخیص اینکه کدام‌یک ارزش زیر نظر گرفتن برای خرید و نگهداری دارند

هدف:
 نوشتن برنامه‌ای که داده‌های واقعی چند سهم را از سایت
tsetmc بگیرد، آن‌ها را تحلیل کند و یک گزارش تمیز تحویل دهد.

چهارچوب پروژه:
----------------------------
1. گرفتن داده: 
 دریافت تاریخچه ۳۰ روزه قیمت و داده‌های حقیقی/حقوقی امروز.

2. تمیزکاری داده (تله‌ها): 
 تبدیل متن به عدد، جدا کردن ریال و تومان، پیدا کردن .کد نماد (insCode) و حل مشکل حروف عربی «ی» و «ک».

3. محاسبات  (فقط با حلقه و لیست، بدون Pandas):
 - تشخیص رنگ کندل (سبز، قرمز، دوجی) و محاسبه درصد بازده روز.
 - محاسبه سقف قیمت ۵ روز اخیر (هفته) و فاصله قیمت فعلی تا آن.
 - محاسبه میانگین حجم ۲۰ روزه و تشخیص حجم غیرعادی (بیشتر از ۲ برابر میانگین).
 - محاسبه ورود/خروج پول حقیقی (خرید حقیقی منهای فروش حقیقی).

4. قانون سیگنال:
 امتیازدهی بر اساس ۴ نشانه و تبدیل آن به کلمات BUY، WATCH یا WAIT.

5. گزارش‌گیری: 
 مرتب‌سازی نتایج از بیشترین امتیاز به کمترین و ذخیره در فایل report.txt با ابزار save_report.

# 1. آماده سازی ابزار

In [1]:
import requests
import time
from urllib.parse import quote

_DIGIT_TRANSLATION = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩",
    "01234567890123456789",
)
_SYMBOL_TRANSLATION = str.maketrans({
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
    "‌": "",
})

def normalize_symbol(value):
    """یکسان‌سازی حروف عربی و فارسی در نام نماد"""
    if value is None:
        return ""
    return str(value).translate(_SYMBOL_TRANSLATION).strip()

def to_number(value):
    """تبدیل عدد متنی فارسی/عربی و جداکننده‌های هزارگان به عدد"""
    if value is None:
        raise ValueError("مقدار عددی خالی است.")
    if isinstance(value, (int, float)):
        return value

    text = str(value).translate(_DIGIT_TRANSLATION).strip()
    text = text.replace("٬", "").replace("،", "").replace(",", "")
    text = text.replace("٫", ".").replace("−", "-")
    text = text.replace("ریال", "").replace("تومان", "").replace(" ", "")
    if not text:
        raise ValueError("مقدار عددی خالی است.")
    if "." in text or "e" in text.lower():
        return float(text)
    return int(text)

def get_json(url):
    """گرفتن داده از API و تبدیل به دیکشنری/لیست پایتون"""
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

def save_report(lines, filename):
    """ذخیره لیست خطوط در یک فایل متنی"""
    with open(filename, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")
    print(f"گزارش با موفقیت در فایل {filename} ذخیره شد.")

print("ابزارها آماده شدند.")

ابزارها آماده شدند.


# 2. پیدا کردن کد نمادها

In [2]:
# لیست ۱۰ نماد برای بررسی
symbols_list = [
    "فولاد", "فملی", "وبملت", "شپنا", "خساپا",
    "فارس", "شتران", "وبصادر", "کگل", "ومعادن",
]
codes = {}

print("در حال دریافت کد نمادها از tsetmc...")
for raw_symbol in symbols_list:
    symbol = normalize_symbol(raw_symbol)
    search_url = (
        "https://cdn.tsetmc.com/api/Instrument/GetInstrumentSearch/"
        + quote(symbol, safe="")
    )
    try:
        data = get_json(search_url)
        candidates = data.get("instrumentSearch", [])

        for item in candidates:
            found_name = normalize_symbol(item.get("lVal18AFC", ""))
            if found_name == symbol and item.get("insCode") is not None:
                codes[symbol] = str(item["insCode"])
                break

        if symbol not in codes:
            print(f"کد نماد {symbol} پیدا نشد.")
    except Exception as error:
        print(f"خطا در جستجوی {symbol}: {error}")

    time.sleep(1)  # رعایت ادب API

print(f"تعداد {len(codes)} نماد پیدا شد.")

در حال دریافت کد نمادها از tsetmc...
تعداد 10 نماد پیدا شد.


# 3. محاسبات اصلی (کندل، بازده، سقف هفته، حجم، پول حقیقی)

In [3]:
results = []

print("\nدر حال دریافت و پردازش داده‌های امروز...")
for symbol, code in codes.items():
    try:
        # دریافت تاریخچه ۳۰ روزه قیمت پایانی
        history_data = get_json(
            f"https://cdn.tsetmc.com/api/ClosingPrice/"
            f"GetClosingPriceDailyList/{code}/30"
        )
        days = history_data.get("closingPriceDaily", [])

        # برای محاسبه میانگین، ۲۰ روز کامل قبل از امروز لازم است
        if len(days) < 21:
            raise ValueError("برای میانگین حجم، حداقل ۲۱ روز داده لازم است.")

        today = days[0]
        today_date = str(today.get("dEven", ""))

        close = int(to_number(today["pClosing"]))
        open_price = int(to_number(today["priceFirst"]))
        yesterday_close = int(to_number(today["priceYesterday"]))
        today_volume = int(to_number(today["qTotTran5J"]))

        if close <= 0 or yesterday_close <= 0:
            raise ValueError("قیمت پایانی امروز یا روز قبل نامعتبر است.")

        # ۱. رنگ کندل و بازده روزانه
        if close > open_price:
            candle = "GREEN"
        elif close < open_price:
            candle = "RED"
        else:
            candle = "DOJI"
        daily_return = ((close - yesterday_close) / yesterday_close) * 100

        # ۲. سقف قیمت پنج روز معاملاتی اخیر و فاصله تا سقف
        recent_five_days = days[:5]
        week_high = int(to_number(recent_five_days[0]["priceMax"]))
        for day in recent_five_days:
            day_high = int(to_number(day["priceMax"]))
            if day_high > week_high:
                week_high = day_high
        gap_to_high = ((week_high - close) / close) * 100

        # ۳. حجم امروز نسبت به میانگین ۲۰ روز معاملاتی قبل
        previous_twenty_days = days[1:21]
        total_volume = 0
        for day in previous_twenty_days:
            total_volume += int(to_number(day["qTotTran5J"]))
        average_volume = total_volume / 20
        unusual_volume = today_volume > (2 * average_volume)

        # ۴. ورود/خروج پول حقیقی؛ مقدار API ریال است و گزارش به تومان
        client_response = get_json(
            f"https://cdn.tsetmc.com/api/ClientType/"
            f"GetClientTypeHistory/{code}"
        )
        client_days = client_response.get("clientType", [])
        if not client_days:
            raise ValueError("داده حقیقی/حقوقی نماد موجود نیست.")

        client_today = client_days[0]
        for client_day in client_days:
            if str(client_day.get("dEven", "")) == today_date:
                client_today = client_day
                break

        net_individual_money_riyal = int(
            to_number(client_today["buy_I_Value"])
            - to_number(client_today["sell_I_Value"])
        )
        net_individual_money_toman = (
            (abs(net_individual_money_riyal) // 10)
            * (-1 if net_individual_money_riyal < 0 else 1)
        )

        results.append({
            "symbol": symbol,
            "date": today_date,
            "close": close,
            "change": daily_return,
            "candle": candle,
            "week_high": week_high,
            "gap_to_high": gap_to_high,
            "today_volume": today_volume,
            "average_volume": average_volume,
            "vol_alert": unusual_volume,
            "money_in": net_individual_money_toman,
        })

    except Exception as error:
        print(f"خطا در پردازش {symbol}: {error}")

    time.sleep(1)

print(f"پردازش داده‌ها به پایان رسید؛ {len(results)} نماد آماده گزارش است.")


در حال دریافت و پردازش داده‌های امروز...
پردازش داده‌ها به پایان رسید؛ 10 نماد آماده گزارش است.


# ۴. قانون سیگنال (امتیازدهی و تشخیص خرید/فروش)

In [4]:
for result in results:
    score = 0

    # ۴ نشانه اصلی
    if result["candle"] == "GREEN":
        score += 1
    if result["money_in"] > 0:
        score += 1
    if result["vol_alert"] and result["candle"] == "GREEN":
        score += 1
    if result["gap_to_high"] < 2:
        score -= 1

    if score >= 2:
        signal = "BUY"
    elif score == 1:
        signal = "WATCH"
    else:
        signal = "WAIT"

    result["score"] = score
    result["signal"] = signal

print("امتیازدهی و سیگنال‌دهی انجام شد.")

امتیازدهی و سیگنال‌دهی انجام شد.


# ۵. مرتب‌سازی و تولید گزارش 

In [5]:
# مرتب‌سازی نتایج از بیشترین امتیاز به کمترین
results.sort(key=lambda result: result["score"], reverse=True)

report_date = results[0]["date"] if results else "نامشخص"
report_lines = [
    f"دیده‌بان نماد - تاریخ: {report_date}",
    "خالص خرید حقیقی از تفاضل ارزش خرید و فروش حقیقی به دست آمده؛ "
    "واحد API ریال و واحد گزارش تومان است.",
    "",
    (
        f"{'سیگنال':<8} | {'امتیاز':>5} | {'نماد':<8} | "
        f"{'پایانی (ریال)':>15} | {'بازده روزانه':>12} | {'کندل':<6} | "
        f"{'سقف ۵ روز (ریال)':>17} | {'فاصله تا سقف':>12} | "
        f"{'حجم امروز':>14} | {'میانگین حجم ۲۰روزه':>19} | "
        f"{'حجم غیرعادی':<12} | {'پول حقیقی (تومان)':>20}"
    ),
    "-" * 170,
]

for result in results:
    volume_label = "بله" if result["vol_alert"] else "خیر"
    line = (
        f"{result['signal']:<8} | {result['score']:>5} | "
        f"{result['symbol']:<8} | {result['close']:>15,} | "
        f"{result['change']:>11.2f}% | {result['candle']:<6} | "
        f"{result['week_high']:>17,} | "
        f"{result['gap_to_high']:>11.2f}% | "
        f"{result['today_volume']:>14,} | "
        f"{result['average_volume']:>19,.0f} | "
        f"{volume_label:<12} | {result['money_in']:>20,}"
    )
    report_lines.append(line)

report_lines.append("")
report_lines.append("تعداد نمادها بر اساس سیگنال:")
for signal_name in ["BUY", "WATCH", "WAIT"]:
    signal_count = 0
    for result in results:
        if result["signal"] == signal_name:
            signal_count += 1
    report_lines.append(f"{signal_name}: {signal_count}")

# خروجی نهایی طبق صورت پروژه در report.txt ذخیره می‌شود
filename = "report.txt"
save_report(report_lines, filename)

for line in report_lines:
    print(line)

گزارش با موفقیت در فایل report.txt ذخیره شد.
دیده‌بان نماد - تاریخ: 20261111
خالص خرید حقیقی از تفاضل ارزش خرید و فروش حقیقی به دست آمده؛ واحد API ریال و واحد گزارش تومان است.

سیگنال   | امتیاز | نماد     |   پایانی (ریال) | بازده روزانه | کندل   |  سقف ۵ روز (ریال) | فاصله تا سقف |      حجم امروز |  میانگین حجم ۲۰روزه | حجم غیرعادی  |    پول حقیقی (تومان)
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
WATCH    |     1 | فملی     |          28,540 |       -2.09% | RED    |            30,150 |        5.64% |    788,411,433 |         511,918,754 | خیر          |       43,466,431,777
WATCH    |     1 | خساپا    |             615 |        2.84% | GREEN  |               615 |        0.00% |  1,932,007,897 |       2,870,280,313 | خیر          |       62,104,435,870
WAIT     |     0 | فولاد    |           4,950 |        2.91% | DOJI   |             4,950 |        0.00% 

## ۷. ماموریت‌های امتیازی

In [6]:
print("\n--- شروع ماموریت‌های امتیازی (بخش API) ---")

# ماموریت ۱: چاپ کلیدهای گواهی سپرده از tgju
print("\n[ماموریت ۱] کلیدهای ime_certificate:")
try:
    current_tgju = get_json("https://call1.tgju.org/ajax.json")["current"]
    for key in current_tgju:
        if "ime_certificate" in key:
            print(f" - {key}: {current_tgju[key]['p']}")
except:
    print("خطا در دریافت داده tgju")

# ماموریت ۲: میانگین قیمت تتر از ۴ صرافی
print("\n[ماموریت ۲] میانگین قیمت تتر:")
tether_prices = []
try: tether_prices.append(int(get_json("https://apiv2.nobitex.ir/market/stats?srcCurrency=usdt&dstCurrency=rls")["stats"]["usdt-rls"]["latest"]) // 10)
except: pass
try:
    markets = get_json("https://api.wallex.ir/hector/web/v1/markets?version=2")["result"]["markets"]
    for m in markets:
        if m["symbol"] == "USDTTMN": tether_prices.append(int(float(m["price"]))); break
except: pass
try:
    rows = get_json("https://api.bitpin.ir/v4/mkt/prices/")
    for row in rows:
        if row["code"] == "USDT_IRT": tether_prices.append(int(float(row["price"]))); break
except: pass
try: tether_prices.append(get_json("https://service.tetherland.com/api/v4/currencies")["price"])
except: pass

if tether_prices:
    print(f"میانگین تتر: {sum(tether_prices)/len(tether_prices):,.0f} تومان")

# ماموریت ۳: انحراف معیار سکه بهار با حلقه
print("\n[ماموریت ۳] انحراف معیار ۳۰ روزه سکه بهار:")
try:
    rows = get_json("https://api.tgju.org/v1/market/indicator/summary-table-data/sekeb?length=30")["data"]
    prices = [int(row[3].replace(",", "")) for row in rows]
    returns = [((prices[i] - prices[i+1]) / prices[i+1]) * 100 for i in range(len(prices)-1)]
    mean = sum(returns) / len(returns)
    variance = sum([(r - mean) ** 2 for r in returns]) / (len(returns) - 1)
    print(f"انحراف معیار: {variance ** 0.5:.2f}%")
except:
    print("خطا در محاسبه انحراف معیار")


--- شروع ماموریت‌های امتیازی (بخش API) ---

[ماموریت ۱] کلیدهای ime_certificate:
 - ime_certificate_silver_bars: 4,400,010
 - ime_certificate_gold_bars: 32,969,890
 - ime_certificate_coin: 2,503,000,300

[ماموریت ۲] میانگین قیمت تتر:
میانگین تتر: 262,201 تومان

[ماموریت ۳] انحراف معیار ۳۰ روزه سکه بهار:
انحراف معیار: 1.45%


### 7. توضیح قانون سیگنال:

### توضیح قانون سیگنال و انتخاب اعداد

**قانون سیگنال من:**
- کندل سبز (بسته شدن بالاتر از بازگشایی): **+۱ امتیاز**
- ورود پول حقیقی مثبت (خریداران قوی‌تر از فروشندگان): **+۱ امتیاز**
- حجم غیرعادی (بیش از ۲ برابر میانگین ۲۰ روزه) همراه با کندل سبز: **+۱ امتیاز**
- فاصله کمتر از ۲ درصد تا سقف هفته (نزدیک به مقاومت): **-۱ امتیاز**

**دلیل انتخاب اعداد:**
- **حجم غیرعادی ۲ برابر:** در تحلیل تکنیکال، حجمی که دو برابر میانگین باشد نشانه ورود پول هوشمند یا یک اتفاق مهم است. اگر این حجم با کندل سبز همراه شود، نشان‌دهنده قدرت خریداران است.
- **فاصله ۲ درصد تا سقف هفته:** اگر قیمت خیلی به سقف ۵ روز اخیر نزدیک باشد (کمتر از ۲٪)، احتمال برخورد با مقاومت و برگشت قیمت وجود دارد، بنابراین یک امتیاز منفی در نظر گرفته می‌شود.
- **آستانه‌های سیگنال:**
  - **BUY:** امتیاز ۲ یا بیشتر (یعنی حداقل دو نشانه مثبت قوی با هم).
  - **WATCH:** امتیاز ۱ (یک نشانه مثبت).
  - **WAIT:** امتیاز ۰ یا کمتر (هیچ نشانه مثبتی وجود ندارد یا در نزدیکی مقاومت است).
  - **QUEUE:** اگر قیمت سقف، کف و پایانی برابر باشد و بازده مثبت باشد، یعنی صف خرید است و اولویت آن از سایر سیگنال‌ها بالاتر است.

##  ۹. بک‌تست استراتژی روی ۳۰ روز گذشته:

In [7]:
print("\n--- شروع بک‌تست استراتژی روی ۳۰ روز گذشته (فقط برای فولاد) ---")

# برای سادگی، بک‌تست را فقط روی نماد اول (مثلاً فولاد) انجام می‌دهیم
target_symbol = list(codes.keys())[0]
target_code = codes[target_symbol]

# دریافت ۴۰ روز داده (۳۰ روز برای بک‌تست + ۱۰ روز برای محاسبه میانگین حجم ۲۰ روزه)
days_bt = get_json(f"https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceDailyList/{target_code}/40")["closingPriceDaily"]

wins = 0
trades = 0

# از ایندکس ۲۰ به بعد شروع می‌کنیم چون به ۲۰ روز داده قبلی برای میانگین حجم نیاز داریم
for i in range(len(days_bt) - 21, 0, -1): 
    t = days_bt[i] # روزی که می‌خواهیم سیگنال آن را بررسی کنیم
    
    # ۱. محاسبه نشانه‌ها برای روز t
    close = int(t["pClosing"])
    first = int(t["priceFirst"])
    color = "GREEN" if close > first else "RED"
    
    # محاسبه میانگین حجم ۲۰ روز قبل از روز t
    total_vol = sum([w["qTotTran5J"] for w in days_bt[i+1 : i+21]])
    avg_vol = total_vol / 20
    vol_alert = t["qTotTran5J"] > (2 * avg_vol)
    
    # فرض می‌کنیم پول حقیقی مثبت است (برای سادگی در بک‌تست، چون تاریخچه حقیقی را نگرفتیم)
    money_in_positive = True 
    
    # ۲. قانون سیگنال (بدون فاصله تا سقف برای سادگی)
    score = 0
    if color == "GREEN": score += 1
    if money_in_positive: score += 1
    if vol_alert and color == "GREEN": score += 1
    
    signal = "BUY" if score >= 2 else "WATCH"
    
    # ۳. بررسی نتیجه (فردای آن روز چه شد؟)
    # چون داده‌ها از جدید به قدیم هستند، روز بعد از t، در ایندکس i-1 قرار دارد
    if signal == "BUY":
        trades += 1
        next_day_close = int(days_bt[i-1]["pClosing"])
        # اگر قیمت پایانی فردا از قیمت پایانی امروز بیشتر بود، یعنی معامله برنده بوده
        if next_day_close > close:
            wins += 1

if trades > 0:
    win_rate = (wins / trades) * 100
    print(f"نماد: {target_symbol}")
    print(f"تعداد سیگنال‌های BUY در ۳۰ روز گذشته: {trades}")
    print(f"تعداد معاملات برنده: {wins}")
    print(f"درصد موفقیت (Win Rate): {win_rate:.2f}%")
else:
    print("در ۳۰ روز گذشته هیچ سیگنال BUY صادر نشده است.")


--- شروع بک‌تست استراتژی روی ۳۰ روز گذشته (فقط برای فولاد) ---
نماد: فولاد
تعداد سیگنال‌های BUY در ۳۰ روز گذشته: 8
تعداد معاملات برنده: 7
درصد موفقیت (Win Rate): 87.50%


## 9. توضیح بک‌تست و مدیریت ریسک:

### نتایج بک‌تست و مدیریت ریسک

در بخش بک‌تست، قانون سیگنال خود را روی ۳۰ روز گذشته یک نماد شاخص (مانند فولاد) آزمایش کردیم.
- **نتیجه بک‌تست:** این آزمایش نشان می‌دهد که قانون ما در گذشته چقدر موفق بوده است. (اگر درصد موفقیت بالای ۵۰٪ باشد، یعنی استراتژی برتری نسبت به شانس دارد).
- **مدیریت ریسک:** در دنیای واقعی، حتی بهترین استراتژی‌ها هم ضرر می‌دهند. یک معامله‌گر حرفه‌ای هرگز بیشتر از ۱ تا ۲ درصد کل سرمایه خود را در یک معامله (مثلاً خرید یک سهم) ریسک نمی‌کند. یعنی اگر قیمت به ضرر رفت (حد ضرر)، فقط ۱ درصد سرمایه از دست می‌رود، نه کل آن.